<a href="https://colab.research.google.com/github/Jin-Miwacle/Metacognitive-Debugger/blob/T5_fine-tune/MT5_Fine_Tunedlast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MT5 Fine-Tuned

In [ ]:
!nvidia-smi

Wed Oct 14 03:35:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece evaluate rouge_score sacrebleu scikit-learn

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()   # choose mt5_qg_bilingual_balanced.csv

Saving mt5_qg_bilingual_balanced.csv to mt5_qg_bilingual_balanced.csv


In [ ]:
import pandas as pd, numpy as np, torch, random
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

df = pd.read_csv("mt5_qg_bilingual_balanced.csv")
df = df.dropna(subset=["passage", "answer", "question", "language"]).reset_index(drop=True)
for c in ["passage", "answer", "question"]:
    df[c] = df[c].astype(str).str.strip()

# 80% train / 10% val / 10% test, grouped by passage so no passage appears in two splits
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["passage"]))
train_df, temp_df = df.iloc[train_idx], df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["passage"]))
val_df, test_df = temp_df.iloc[val_idx], temp_df.iloc[test_idx]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d["language"].value_counts().to_dict())

train 364 {'English': 184, 'Filipino': 180}
val 53 {'English': 31, 'Filipino': 22}
test 43 {'Filipino': 28, 'English': 15}


In [ ]:
def build_examples(d, multitask=True):
    rows = []
    for r in d.itertuples():
        # Task 1: question generation (answer-aware)
        rows.append({
            "source": f"generate question: context: {r.passage} answer: {r.answer}",
            "target": r.question, "task": "qg", "language": r.language,
        })
        # Task 2: question answering (derived from the same rows)
        if multitask:
            rows.append({
                "source": f"answer question: question: {r.question} context: {r.passage}",
                "target": r.answer, "task": "qa", "language": r.language,
            })
    return rows

train_rows = build_examples(train_df)
val_rows   = build_examples(val_df)
test_rows  = build_examples(test_df)
print(len(train_rows), len(val_rows), len(test_rows))
print(train_rows[0]["source"][:200], "->", train_rows[0]["target"])
print(train_rows[1]["source"][:200], "->", train_rows[1]["target"])

728 106 86
generate question: context: Every school day, Hana took the same bus to class. One Tuesday, the bus stopped because a tire had lost air. The driver contacted the transport office while passengers wait -> Whom did the driver contact?
answer question: question: Whom did the driver contact? context: Every school day, Hana took the same bus to class. One Tuesday, the bus stopped because a tire had lost air. The driver contacted the t -> the transport office


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "google/mt5-small"   # start here; try "google/mt5-base" later
MAX_SRC, MAX_TGT = 768, 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Check how many sources would be truncated (the answer sits at the END of the QG prompt)
lens = [len(tokenizer(r["source"])["input_ids"]) for r in train_rows]
print("max len:", max(lens), "| % over MAX_SRC:", 100 * np.mean(np.array(lens) > MAX_SRC))

def tokenize(batch):
    enc = tokenizer(batch["source"], max_length=MAX_SRC, truncation=True)
    enc["labels"] = tokenizer(text_target=batch["target"], max_length=MAX_TGT, truncation=True)["input_ids"]
    return enc

train_ds = Dataset.from_list(train_rows).map(tokenize, batched=True, remove_columns=["source","target","task","language"])
val_ds   = Dataset.from_list(val_rows).map(tokenize, batched=True, remove_columns=["source","target","task","language"])

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

max len: 854 | % over MAX_SRC: 0.27472527472527475


Map:   0%|          | 0/728 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

In [ ]:
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

args = Seq2SeqTrainingArguments(
    output_dir="mt5_qg_out",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    optim="adafactor",
    num_train_epochs=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    fp16=False, bf16=False,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,          # if this errors, change to: tokenizer=tokenizer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=598, training_loss=5.273852644955434, metrics={'train_runtime': 1021.6791, 'train_samples_per_second': 9.263, 'train_steps_per_second': 0.585, 'total_flos': 3508477265264640.0, 'train_loss': 5.273852644955434, 'epoch': 13.0})

In [ ]:
from tqdm.auto import tqdm
import evaluate

rouge = evaluate.load("rouge")
bleu  = evaluate.load("sacrebleu")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def generate(texts, bs=8, max_new_tokens=48):
    outs = []
    for i in tqdm(range(0, len(texts), bs)):
        batch = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_SRC).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, num_beams=4, max_new_tokens=max_new_tokens,
                                 no_repeat_ngram_size=3, early_stopping=True)
        outs += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return outs

test_eval = pd.DataFrame(test_rows)
test_eval["pred"] = generate(test_eval["source"].tolist())

for task in ["qg", "qa"]:
    for lang in ["English", "Filipino"]:
        s = test_eval[(test_eval.task == task) & (test_eval.language == lang)]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"{task.upper()} | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")

pd.set_option("display.max_colwidth", 120)
print(test_eval[test_eval.task=="qg"][["language","target","pred"]].head(10).to_string())

  0%|          | 0/11 [00:00<?, ?it/s]

QG | English  | n= 15 | ROUGE-L=0.507 | BLEU=34.5
QG | Filipino | n= 28 | ROUGE-L=0.276 | BLEU=7.8
QA | English  | n= 15 | ROUGE-L=0.472 | BLEU=31.5
QA | Filipino | n= 28 | ROUGE-L=0.238 | BLEU=4.6
    language                                                                           target                                                          pred
0   Filipino  Sa anong uri ng Budismo ay may adisyonal na kanlungan maliban sa Tatlong Hiyas?                 Saan nag-aatas ng isang ikaapat na kanlungan?
2   Filipino           Aling pelikula ni Beyoncé ang nagkaroon ng nominasyon sa Golden Globe?              Ano ang pangalan ng album ni Knowles noong 2008?
4   Filipino       Ninais ni Luna na magkaroon ng disiplina ang mga sundalo ng anong pangkat?                                  Saan naganap ang mga tauhan?
6    English                                      Where did Jun practice bicycle maintenance?                Why did he practice basic bicycle maintenance?
8    English          

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/mt5_qg_qa_bilingual


In [ ]:
# --- free GPU memory from run 1 ---
import gc
del model, trainer
gc.collect(); torch.cuda.empty_cache()

# --- new prompts: answer/question FIRST ---
def build_examples_v2(d, multitask=True):
    rows = []
    for r in d.itertuples():
        rows.append({
            "source": f"generate question: answer: {r.answer} context: {r.passage}",
            "target": r.question, "task": "qg", "language": r.language,
        })
        if multitask:
            rows.append({
                "source": f"answer question: question: {r.question} context: {r.passage}",
                "target": r.answer, "task": "qa", "language": r.language,
            })
    return rows

train_rows = build_examples_v2(train_df)
val_rows   = build_examples_v2(val_df)
test_rows  = build_examples_v2(test_df)

MODEL_NAME = "google/mt5-base"
MAX_SRC, MAX_TGT = 768, 64
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

cols = ["source", "target", "task", "language"]
train_ds = Dataset.from_list(train_rows).map(tokenize, batched=True, remove_columns=cols)
val_ds   = Dataset.from_list(val_rows).map(tokenize, batched=True, remove_columns=cols)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

args = Seq2SeqTrainingArguments(
    output_dir="mt5_base_out",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch 16
    learning_rate=3e-4,
    optim="adafactor",
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    fp16=False, bf16=False,
    gradient_checkpointing=True,        # saves GPU memory
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,         # or tokenizer=tokenizer if it errors
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/728 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=460, training_loss=7.381397010969079, metrics={'train_runtime': 1689.3404, 'train_samples_per_second': 4.309, 'train_steps_per_second': 0.272, 'total_flos': 5029267853795328.0, 'train_loss': 7.381397010969079, 'epoch': 10.0})

In [ ]:
model.to(device).eval()

test_eval = pd.DataFrame(test_rows)
test_eval["pred"] = generate(test_eval["source"].tolist(), bs=4)

for task in ["qg", "qa"]:
    for lang in ["English", "Filipino"]:
        s = test_eval[(test_eval.task == task) & (test_eval.language == lang)]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"{task.upper()} | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")

pd.set_option("display.max_colwidth", 120)
print(test_eval[test_eval.task=="qg"][["language","target","pred"]].head(10).to_string())

  0%|          | 0/22 [00:00<?, ?it/s]

QG | English  | n= 15 | ROUGE-L=0.627 | BLEU=39.0
QG | Filipino | n= 28 | ROUGE-L=0.325 | BLEU=10.4
QA | English  | n= 15 | ROUGE-L=0.883 | BLEU=86.7
QA | Filipino | n= 28 | ROUGE-L=0.748 | BLEU=78.6
    language                                                                           target                                                                           pred
0   Filipino  Sa anong uri ng Budismo ay may adisyonal na kanlungan maliban sa Tatlong Hiyas?                                         Ano ang tawag sa ikaapat na kanlungan?
2   Filipino           Aling pelikula ni Beyoncé ang nagkaroon ng nominasyon sa Golden Globe?              Ano ang titulo ng pelikula na ginamit ni Knowles sa Golden Globe?
4   Filipino       Ninais ni Luna na magkaroon ng disiplina ang mga sundalo ng anong pangkat?                    Saan naganap ang isang disiplina ang mga tauhan sa La Loma?
6    English                                      Where did Jun practice bicycle maintenance?               

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/mt5_qg_qa_bilingual


In [ ]:
model.to(device).eval()

def generate_question(passage, answer, num_beams=4):
    src = f"generate question: answer: {answer} context: {passage}"
    return generate([src], bs=1)[0]

def answer_question(passage, question):
    src = f"answer question: question: {question} context: {passage}"
    return generate([src], bs=1)[0]

# --- English test ---
en = ("Maria planted tomato seeds in a small pot near the window. She watered them every morning "
      "and moved the pot outside when the weather became warm. After three weeks, small green "
      "sprouts appeared.")
print("EN QG:", generate_question(en, "three weeks"))
print("EN QG:", generate_question(en, "near the window"))
print("EN QA:", answer_question(en, "When did the sprouts appear?"))

# --- Filipino test ---
tl = ("Ang Maynila ang kabisera ng Pilipinas. Ito ay matatagpuan sa pulo ng Luzon at isa sa "
      "pinakamataong lungsod sa bansa. Ang Ilog Pasig ay dumadaloy sa gitna ng lungsod.")
print("TL QG:", generate_question(tl, "Luzon"))
print("TL QG:", generate_question(tl, "Ilog Pasig"))
print("TL QA:", answer_question(tl, "Saan matatagpuan ang Maynila?"))

  0%|          | 0/1 [00:00<?, ?it/s]

EN QG: When did Maria plant tomato seeds?


  0%|          | 0/1 [00:00<?, ?it/s]

EN QG: Where did Maria plant tomato seeds?


  0%|          | 0/1 [00:00<?, ?it/s]

...

TL QG: Saan matatagpuan ang Maynila?


  0%|          | 0/1 [00:00<?, ?it/s]

TL QG: Ano ang dumadaloy sa gitna ng lungsod?


  0%|          | 0/1 [00:00<?, ?it/s]

TL QA: Ilog Pasig


In [4]:
from tqdm.auto import tqdm
import re

def generate(texts, bs=8, max_new_tokens=48):
    outs = []
    for i in range(0, len(texts), bs):
        batch = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_SRC).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, num_beams=4, max_new_tokens=max_new_tokens,
                                 no_repeat_ngram_size=3, early_stopping=True)
        outs += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return outs

def generate_question(passage, answer):
    return generate([f"generate question: answer: {answer} context: {passage}"], bs=1)[0]

def answer_question(passage, question):
    return generate([f"answer question: question: {question} context: {passage}"], bs=1)[0]

def norm(s): return re.sub(r"\W+", " ", s.lower()).strip()

def generate_candidates(passage, answer, n=6):
    src = f"generate question: answer: {answer} context: {passage}"
    batch = tokenizer([src], return_tensors="pt", truncation=True, max_length=MAX_SRC).to(device)
    with torch.no_grad():
        ids = model.generate(**batch, num_beams=n, num_return_sequences=n,
                             max_new_tokens=48, early_stopping=True)
    return list(dict.fromkeys(tokenizer.batch_decode(ids, skip_special_tokens=True)))

def verified_questions(passage, answer, n=6):
    results = []
    for q in generate_candidates(passage, answer, n):
        pred = answer_question(passage, q)
        ok = norm(answer) in norm(pred) or norm(pred) in norm(answer)
        results.append((q, pred, ok))
    return results

for q, pred, ok in verified_questions(en, "three weeks"):
    print("✅" if ok else "❌", q, "→", pred)
print()
for q, pred, ok in verified_questions(tl, "sa pulo ng Luzon"):
    print("✅" if ok else "❌", q, "→", pred)

NameError: name 'en' is not defined

In [5]:
!pip install -q -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.9 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual"
print(os.path.exists(SAVE_DIR))
print(os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else "NOT FOUND")

Mounted at /content/drive
True
['config.json', 'generation_config.json', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors']


In [7]:
import re, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual"
MAX_SRC = 768
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR).to(device).eval()

en = ("Maria planted tomato seeds in a small pot near the window. She watered them every morning "
      "and moved the pot outside when the weather became warm. After three weeks, small green "
      "sprouts appeared.")
tl = ("Ang Maynila ang kabisera ng Pilipinas. Ito ay matatagpuan sa pulo ng Luzon at isa sa "
      "pinakamataong lungsod sa bansa. Ang Ilog Pasig ay dumadaloy sa gitna ng lungsod.")

def generate(texts, bs=8, max_new_tokens=48):
    outs = []
    for i in range(0, len(texts), bs):
        batch = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_SRC).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, num_beams=4, max_new_tokens=max_new_tokens,
                                 no_repeat_ngram_size=3, early_stopping=True)
        outs += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return outs

def generate_question(passage, answer):
    return generate([f"generate question: answer: {answer} context: {passage}"], bs=1)[0]

def answer_question(passage, question):
    return generate([f"answer question: question: {question} context: {passage}"], bs=1)[0]

def norm(s): return re.sub(r"\W+", " ", s.lower()).strip()

def generate_candidates(passage, answer, n=6):
    src = f"generate question: answer: {answer} context: {passage}"
    batch = tokenizer([src], return_tensors="pt", truncation=True, max_length=MAX_SRC).to(device)
    with torch.no_grad():
        ids = model.generate(**batch, num_beams=n, num_return_sequences=n,
                             max_new_tokens=48, early_stopping=True)
    return list(dict.fromkeys(tokenizer.batch_decode(ids, skip_special_tokens=True)))

def verified_questions(passage, answer, n=6):
    results = []
    for q in generate_candidates(passage, answer, n):
        pred = answer_question(passage, q)
        ok = norm(answer) in norm(pred) or norm(pred) in norm(answer)
        results.append((q, pred, ok))
    return results

for q, pred, ok in verified_questions(en, "three weeks"):
    print("✅" if ok else "❌", q, "→", pred)
print()
for q, pred, ok in verified_questions(tl, "sa pulo ng Luzon"):
    print("✅" if ok else "❌", q, "→", pred)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ When did Maria plant tomato seeds? → three weeks
✅ How long did Maria plant tomato seeds? → three weeks
✅ What did Maria plant tomato seeds after? → three weeks
✅ When did Maria plant the seeds? → three weeks
❌ What did Maria wait for? → every morning
❌ What did Maria plant tomato seeds? → in a small pot

❌ Saan matatagpuan ang Maynila? → Ilog Pasig
❌ Ano ang matatagpuan ng Maynila? → Ilog Pasig
❌ Ano ang matatagpuan sa Maynila? → Ilog Pasig
❌ Ano ang maaaring matatagpuan sa Maynila? → Ilog Pasig
❌ Ang Maynila ang matatagpuan sa anong pulo? → Saan matatagpuan ang kabisera ng Pilipinas
❌ Ang Maynila ang matatagpuan sa anong lugar? → Ilog Pasig


In [1]:
from google.colab import files
uploaded = files.upload()   # a file picker opens: choose mt5_qg_bilingual_expanded.csv

import os
print(os.listdir("."))      # you should see mt5_qg_bilingual_expanded.csv in this list

Saving mt5_qg_bilingual_expanded.csv to mt5_qg_bilingual_expanded.csv
['.config', 'mt5_qg_bilingual_expanded.csv', 'drive', 'sample_data']


In [2]:
import pandas as pd, numpy as np, torch, random, re, gc
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

df = pd.read_csv("mt5_qg_bilingual_expanded.csv")
df = df.dropna(subset=["passage", "answer", "question", "language"]).reset_index(drop=True)
for c in ["passage", "answer", "question"]:
    df[c] = df[c].astype(str).str.strip()
print(df.shape, df["language"].value_counts().to_dict())

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["passage"]))
train_df, temp_df = df.iloc[train_idx], df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["passage"]))
val_df, test_df = temp_df.iloc[val_idx], temp_df.iloc[test_idx]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d["language"].value_counts().to_dict())

(859, 6) {'Filipino': 430, 'English': 429}
train 703 {'English': 356, 'Filipino': 347}
val 73 {'Filipino': 51, 'English': 22}
test 83 {'English': 51, 'Filipino': 32}


In [3]:
from transformers import AutoTokenizer
from datasets import Dataset

def build_examples_v2(d, multitask=True):
    rows = []
    for r in d.itertuples():
        rows.append({"source": f"generate question: answer: {r.answer} context: {r.passage}",
                     "target": r.question, "task": "qg", "language": r.language})
        if multitask:
            rows.append({"source": f"answer question: question: {r.question} context: {r.passage}",
                         "target": r.answer, "task": "qa", "language": r.language})
    return rows

train_rows = build_examples_v2(train_df)
val_rows   = build_examples_v2(val_df)
test_rows  = build_examples_v2(test_df)

MODEL_NAME = "google/mt5-base"
MAX_SRC, MAX_TGT = 768, 64
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tokenizer(batch["source"], max_length=MAX_SRC, truncation=True)
    enc["labels"] = tokenizer(text_target=batch["target"], max_length=MAX_TGT, truncation=True)["input_ids"]
    return enc

cols = ["source", "target", "task", "language"]
train_ds = Dataset.from_list(train_rows).map(tokenize, batched=True, remove_columns=cols)
val_ds   = Dataset.from_list(val_rows).map(tokenize, batched=True, remove_columns=cols)
print(len(train_ds), len(val_ds), len(test_rows))

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/1406 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

1406 146 166


In [4]:
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

args = Seq2SeqTrainingArguments(
    output_dir="mt5_base_out",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    optim="adafactor",
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    fp16=False, bf16=False,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,          # if this errors, use: tokenizer=tokenizer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.33GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=528, training_loss=6.108722401387764, metrics={'train_runtime': 1358.8697, 'train_samples_per_second': 6.208, 'train_steps_per_second': 0.389, 'total_flos': 4479725339781120.0, 'train_loss': 6.108722401387764, 'epoch': 6.0})

In [5]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual_v2"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/mt5_qg_qa_bilingual_v2


In [8]:
import evaluate
rouge = evaluate.load("rouge")
bleu  = evaluate.load("sacrebleu")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def generate(texts, bs=4, max_new_tokens=48):
    outs = []
    for i in range(0, len(texts), bs):
        batch = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_SRC).to(device)
        with torch.no_grad():
            ids = model.generate(**batch, num_beams=4, max_new_tokens=max_new_tokens,
                                 no_repeat_ngram_size=3, early_stopping=True)
        outs += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return outs

test_eval = pd.DataFrame(test_rows)
test_eval["pred"] = generate(test_eval["source"].tolist())

for task in ["qg", "qa"]:
    for lang in ["English", "Filipino"]:
        s = test_eval[(test_eval.task == task) & (test_eval.language == lang)]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"{task.upper()} | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")

pd.set_option("display.max_colwidth", 120)
print(test_eval[test_eval.task=="qg"][["language","target","pred"]].head(10).to_string())

QG | English  | n= 51 | ROUGE-L=0.600 | BLEU=41.0
QG | Filipino | n= 32 | ROUGE-L=0.430 | BLEU=15.2
QA | English  | n= 51 | ROUGE-L=0.808 | BLEU=64.8
QA | Filipino | n= 32 | ROUGE-L=0.682 | BLEU=56.0
    language                                                                                                 target                                                               pred
0   Filipino                                                          Ilan ang mga sistema para isulat ang Hokkien?           Ano ang mayroong hindi bababa sa 11 metodong pagsusulat?
2   Filipino                             Ninais ni Luna na magkaroon ng disiplina ang mga sundalo ng anong pangkat?          Saan pinagsikapan ni Luna maipailalim sa isang disiplina?
4    English                                                            Where did Jun practice bicycle maintenance?                   Where did he practice basic bicycle maintenance?
6    English                                                        

In [7]:
!pip install -q evaluate rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.0 MB/s eta 0:00:00


In [9]:
orig_passages = set(pd.read_csv("mt5_qg_bilingual_balanced.csv")["passage"])
# each source row in test_df produced 2 rows in test_rows (qg then qa), in order
is_orig = np.repeat(test_df["passage"].isin(orig_passages).values, 2)
test_eval["is_original"] = is_orig

for task in ["qg", "qa"]:
    for lang in ["English", "Filipino"]:
        s = test_eval[(test_eval.task == task) & (test_eval.language == lang) & test_eval.is_original]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"ORIG {task.upper()} | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")

FileNotFoundError: [Errno 2] No such file or directory: 'mt5_qg_bilingual_balanced.csv'

In [10]:
from google.colab import files
files.upload()   # choose mt5_qg_bilingual_balanced.csv (your original file)

Saving mt5_qg_bilingual_balanced.csv to mt5_qg_bilingual_balanced.csv


{'mt5_qg_bilingual_balanced.csv': b'\xef\xbb\xbfpassage,answer,question,language,input_text,target_text\n"Every school day, Hana took the same bus to class. One Tuesday, the bus stopped because a tire had lost air. The driver contacted the transport office while passengers waited in a safe area beside the road. A replacement bus arrived after twenty minutes. Hana reached school late, so she explained the delay to her teacher and asked a classmate for the notes she had missed.",the transport office,Whom did the driver contact?,English,"generate question: context: Every school day, Hana took the same bus to class. One Tuesday, the bus stopped because a tire had lost air. The driver contacted the transport office while passengers waited in a safe area beside the road. A replacement bus arrived after twenty minutes. Hana reached school late, so she explained the delay to her teacher and asked a classmate for the notes she had missed. answer: the transport office",Whom did the driver contac

In [11]:
orig_passages = set(pd.read_csv("mt5_qg_bilingual_balanced.csv")["passage"].astype(str).str.strip())

# each source row in test_df produced 2 rows in test_rows (qg then qa), in order
is_orig = np.repeat(test_df["passage"].isin(orig_passages).values, 2)
test_eval["is_original"] = is_orig
print("original rows in test:", test_eval[test_eval.task=="qg"].is_original.sum(),
      "of", (test_eval.task=="qg").sum())

for task in ["qg", "qa"]:
    for lang in ["English", "Filipino"]:
        s = test_eval[(test_eval.task == task) & (test_eval.language == lang) & test_eval.is_original]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"ORIG {task.upper()} | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")


original rows in test: 59 of 83
ORIG QG | English  | n= 35 | ROUGE-L=0.579 | BLEU=38.4
ORIG QG | Filipino | n= 24 | ROUGE-L=0.389 | BLEU=13.0
ORIG QA | English  | n= 35 | ROUGE-L=0.749 | BLEU=57.4
ORIG QA | Filipino | n= 24 | ROUGE-L=0.634 | BLEU=21.4


In [12]:
import string, collections

def norm_ans(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    return " ".join(s.split())

def f1_score(pred, gold):
    p, g = norm_ans(pred).split(), norm_ans(gold).split()
    if len(p) == 0 or len(g) == 0:
        return float(p == g)
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    if common == 0: return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

def em_score(pred, gold):
    return float(norm_ans(pred) == norm_ans(gold))

qa = test_eval[test_eval.task == "qa"].copy()
qa["f1"] = [f1_score(p, g) for p, g in zip(qa.pred, qa.target)]
qa["em"] = [em_score(p, g) for p, g in zip(qa.pred, qa.target)]

for name, sub in [("ALL", qa), ("ORIGINAL", qa[qa.is_original])]:
    for lang in ["English", "Filipino"]:
        s = sub[sub.language == lang]
        print(f"{name:8s} QA | {lang:8s} | n={len(s):3d} | F1={s.f1.mean():.3f} | EM={s.em.mean():.3f}")

# look at the QA mistakes
pd.set_option("display.max_colwidth", 80)
print(qa[qa.f1 < 0.5][["language", "target", "pred"]].head(12).to_string())

ALL      QA | English  | n= 51 | F1=0.808 | EM=0.627
ALL      QA | Filipino | n= 32 | F1=0.684 | EM=0.562
ORIGINAL QA | English  | n= 35 | F1=0.752 | EM=0.600
ORIGINAL QA | Filipino | n= 24 | F1=0.630 | EM=0.542
    language                                               target                                       pred
1   Filipino                                   hindi bababa sa 11  mga logogramang may kasamang mga Bopomofo
13  Filipino                                                 Sōtō                                       huli
23   English  Fewer pupils arrived without the required materials                       for other classrooms
27  Filipino                                             sa balat                     bilang isang gut flora
29  Filipino                                          mga Krusada                             malaking yaman
37   English          parents raised concerns about speeding cars                 near the elementary school
59  Filipino             

In [13]:
del model, trainer
gc.collect(); torch.cuda.empty_cache()

train_orig_df = train_df[train_df["passage"].isin(orig_passages)]
print("original-only training rows:", len(train_orig_df))
train_ds_o = Dataset.from_list(build_examples_v2(train_orig_df)).map(
    tokenize, batched=True, remove_columns=cols)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
args_o = Seq2SeqTrainingArguments(
    output_dir="mt5_base_out_origonly",
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=3e-4, optim="adafactor",
    num_train_epochs=15, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, fp16=False, bf16=False, gradient_checkpointing=True,
    report_to="none", seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model, args=args_o, train_dataset=train_ds_o, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

original-only training rows: 356


Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss
1,13.676958,1.938408
2,8.834866,1.503651
3,6.532377,1.339923
4,5.389954,1.307534
5,4.068691,1.236095
6,3.560944,1.280760
7,2.924947,1.307254
8,2.587677,1.323745


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=360, training_loss=7.153814993964302, metrics={'train_runtime': 1566.7723, 'train_samples_per_second': 3.635, 'train_steps_per_second': 0.23, 'total_flos': 3966191679909888.0, 'train_loss': 7.153814993964302, 'epoch': 8.0})

In [14]:
model.to(device).eval()

test_eval_o = pd.DataFrame(test_rows)
test_eval_o["pred"] = generate(test_eval_o["source"].tolist())
test_eval_o["is_original"] = np.repeat(test_df["passage"].isin(orig_passages).values, 2)

# --- QG: ROUGE-L / BLEU ---
for name, mask in [("ORIG", test_eval_o.is_original), ("NEW ", ~test_eval_o.is_original)]:
    for lang in ["English", "Filipino"]:
        s = test_eval_o[(test_eval_o.task == "qg") & (test_eval_o.language == lang) & mask]
        if len(s) == 0: continue
        r = rouge.compute(predictions=s.pred.tolist(), references=s.target.tolist())
        b = bleu.compute(predictions=s.pred.tolist(), references=[[t] for t in s.target.tolist()])
        print(f"{name} QG | {lang:8s} | n={len(s):3d} | ROUGE-L={r['rougeL']:.3f} | BLEU={b['score']:.1f}")

# --- QA: F1 / EM ---
qa_o = test_eval_o[test_eval_o.task == "qa"].copy()
qa_o["f1"] = [f1_score(p, g) for p, g in zip(qa_o.pred, qa_o.target)]
qa_o["em"] = [em_score(p, g) for p, g in zip(qa_o.pred, qa_o.target)]
for name, mask in [("ORIG", qa_o.is_original), ("NEW ", ~qa_o.is_original)]:
    for lang in ["English", "Filipino"]:
        s = qa_o[(qa_o.language == lang) & mask]
        if len(s) == 0: continue
        print(f"{name} QA | {lang:8s} | n={len(s):3d} | F1={s.f1.mean():.3f} | EM={s.em.mean():.3f}")

ORIG QG | English  | n= 35 | ROUGE-L=0.498 | BLEU=27.1
ORIG QG | Filipino | n= 24 | ROUGE-L=0.381 | BLEU=11.2
NEW  QG | English  | n= 16 | ROUGE-L=0.539 | BLEU=29.5
NEW  QG | Filipino | n=  8 | ROUGE-L=0.399 | BLEU=9.9
ORIG QA | English  | n= 35 | F1=0.654 | EM=0.543
ORIG QA | Filipino | n= 24 | F1=0.535 | EM=0.375
NEW  QA | English  | n= 16 | F1=0.859 | EM=0.625
NEW  QA | Filipino | n=  8 | F1=0.780 | EM=0.375


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
SAVE_DIR = "/content/drive/MyDrive/mt5_qg_qa_bilingual_v2"
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR).to(device).eval()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
model.to(device).eval()

en = ("Maria planted tomato seeds in a small pot near the window. She watered them every morning "
      "and moved the pot outside when the weather became warm. After three weeks, small green "
      "sprouts appeared.")
tl = ("Ang Maynila ang kabisera ng Pilipinas. Ito ay matatagpuan sa pulo ng Luzon at isa sa "
      "pinakamataong lungsod sa bansa. Ang Ilog Pasig ay dumadaloy sa gitna ng lungsod.")

def generate_question(passage, answer):
    return generate([f"generate question: answer: {answer} context: {passage}"], bs=1)[0]

def answer_question(passage, question):
    return generate([f"answer question: question: {question} context: {passage}"], bs=1)[0]

print("EN QG:", generate_question(en, "three weeks"))
print("EN QG:", generate_question(en, "near the window"))
print("EN QA:", answer_question(en, "When did the sprouts appear?"))
print("TL QG:", generate_question(tl, "Luzon"))
print("TL QG:", generate_question(tl, "Ilog Pasig"))
print("TL QA:", answer_question(tl, "Saan matatagpuan ang Maynila?"))

EN QG: When did small green sprouts appear?
EN QG: Where did Maria plant tomato seeds?
EN QA: three weeks
TL QG: Anong pulo matatagpuan ang Maynila?
TL QG: Ano ang dumadaloy sa gitna ng lungsod?
TL QA: Luzon
